In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os

# ---- Paths ----
DATA_PATH = "data/transit_ridership.csv"
SUMMARY_PATH = "output/summary.json"
CHART_DIR = "output/charts"

os.makedirs(CHART_DIR, exist_ok=True)  # ensure charts folder exists

# ---- 1. Load CSV ----
df = pd.read_csv(DATA_PATH)

# ---- 2. Cleaning Function ----
def clean_data(df):
    # Convert date
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Fill missing numeric values
    df['boarding_count'] = df['boarding_count'].fillna(0)
    df['alighting_count'] = df['alighting_count'].fillna(0)
    df['trip_duration_min'] = df['trip_duration_min'].fillna(df['trip_duration_min'].median())
    df['temperature_c'] = df['temperature_c'].fillna(df['temperature_c'].median())
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Clean text columns
    for col in ['direction', 'vehicle_type', 'weather']:
        df[col] = df[col].str.strip().str.title()
    
    # Standardize 'is_holiday' to boolean
    df['is_holiday'] = df['is_holiday'].astype(str).str.lower().map({
        'false': False, '0': False, 'no': False,
        'true': True, '1': True, 'yes': True
    }).fillna(False)
    
    # Remove invalid numeric entries
    df = df[(df['boarding_count'] >= 0) & (df['alighting_count'] >= 0)]
    df = df[df['trip_duration_min'] >= 0]
    df = df[df['temperature_c'].between(-50, 60)]
    
    df = df.reset_index(drop=True)
    return df

df_clean = clean_data(df)

# ---- 3. Summary Statistics ----
# ---- Generate summary.json in the required format ----

def generate_summary(df):
    total_trips = len(df)
    # use 'date' column instead of index
    date_range = f"{df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}"
    
    route_boardings = df.groupby('route_id')['boarding_count'].sum()
    busiest_route = route_boardings.idxmax()
    
    daily_totals = df.groupby(df['date'].dt.date)[['boarding_count', 'alighting_count']].sum()
    avg_daily_ridership = round(daily_totals.sum(axis=1).mean(), 1)
    
    ridership_by_vehicle_type = df.groupby('vehicle_type')['boarding_count'].sum().to_dict()
    ridership_by_weather = df.groupby('weather')['boarding_count'].sum().to_dict()
    
    top_routes = route_boardings.sort_values(ascending=False).head(5)
    top_5_routes_by_boarding = [{"route": r, "total_boardings": int(b)} for r, b in top_routes.items()]
    
    summary = {
        "total_trips": total_trips,
        "date_range": date_range,
        "busiest_route": busiest_route,
        "avg_daily_ridership": avg_daily_ridership,
        "ridership_by_vehicle_type": {k: int(v) for k,v in ridership_by_vehicle_type.items()},
        "ridership_by_weather": {k: int(v) for k,v in ridership_by_weather.items()},
        "top_5_routes_by_boarding": top_5_routes_by_boarding
    }
    
    return summary
# ---- 5. Charts ----
# ---- Monthly aggregation for charts ----
df_clean.set_index('date', inplace=True)
monthly = df_clean.resample('ME').agg({
    'boarding_count': 'sum',
    'alighting_count': 'sum',
    'trip_duration_min': 'mean',
    'temperature_c': 'mean'
})

# Chart 1: Boarding vs Alighting
plt.figure(figsize=(12,6))
plt.plot(monthly.index, monthly['boarding_count'], marker='o', label='Boarding')
plt.plot(monthly.index, monthly['alighting_count'], marker='o', label='Alighting')
plt.title('Monthly Ridership')
plt.xlabel('Month')
plt.ylabel('Number of Passengers')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/monthly_ridership.png")
plt.close()

# Chart 2: Average Trip Duration per Month
plt.figure(figsize=(12,6))
plt.plot(monthly.index, monthly['trip_duration_min'], marker='o', color='orange')
plt.title('Average Trip Duration per Month')
plt.xlabel('Month')
plt.ylabel('Duration (min)')
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/avg_trip_duration.png")
plt.close()

# Chart 3: Average Temperature per Month
plt.figure(figsize=(12,6))
plt.plot(monthly.index, monthly['temperature_c'], marker='o', color='green')
plt.title('Average Temperature per Month')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/avg_temperature.png")
plt.close()

# Chart 4: Vehicle Type Counts (Pie Chart)
vehicle_counts = df_clean['vehicle_type'].value_counts()
plt.figure(figsize=(12,8))
plt.pie(vehicle_counts, labels=vehicle_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Vehicle Type Distribution')
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/vehicle_type_distribution.png")
plt.close()

print(f"Charts saved to {CHART_DIR}")

Charts saved to output/charts
